# ПР-8. Градиентный бустинг

Изучить последовательное исправление ошибок, сравнить бустинг со случайным лесом и увидеть роль ранней остановки/числа деревьев.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('train/test:', X_train.shape, X_test.shape)


## 1. Обучение и сравнение

Сравните Random Forest и Gradient Boosting на одинаковом train/test-разбиении. Для подбора используйте только train и стратифицированную CV.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
 'Random Forest': GridSearchCV(RandomForestClassifier(random_state=42),
     {'n_estimators':[100,300], 'max_depth':[None,6], 'min_samples_leaf':[1,3]},
     scoring='f1_macro', cv=cv, n_jobs=-1),
 'Gradient Boosting': GridSearchCV(GradientBoostingClassifier(random_state=42),
     {'n_estimators':[50,100,200], 'learning_rate':[0.03,0.1], 'max_depth':[1,2,3]},
     scoring='f1_macro', cv=cv, n_jobs=-1)
}
for name, search in models.items():
    search.fit(X_train, y_train)
    pred=search.predict(X_test)
    print(name, search.best_params_, 'CV F1macro=', round(search.best_score_,3),
          'test F1macro=', round(f1_score(y_test,pred,average='macro'),3),
          'test accuracy=', round(accuracy_score(y_test,pred),3))
    print(classification_report(y_test,pred,target_names=['malignant','benign']))

## 2. Кривая числа деревьев

Оцените качество по мере добавления слабых деревьев. Тест используйте только для финальной оценки; кривую строим на обучающих данных через CV.

In [ ]:
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
counts=[10,25,50,100,200]
means=[]
for n in counts:
    m=GradientBoostingClassifier(n_estimators=n, learning_rate=0.05, max_depth=2, random_state=42)
    means.append(cross_val_score(m,X_train,y_train,cv=cv,scoring='f1_macro',n_jobs=-1).mean())
plt.plot(counts,means,marker='o'); plt.xlabel('n_estimators'); plt.ylabel('CV F1 macro'); plt.grid(True); plt.show()

## Контрольные вопросы и вывод

1. В чём отличие последовательного обучения бустинга от бэггинга случайного леса?
2. Как менялся CV F1 при увеличении n_estimators? Был ли участок, где улучшение стало небольшим?
3. Почему нельзя выбирать число деревьев по тестовой метрике?
4. Назовите риск бустинга на зашумлённых данных.

**В отчёте:** таблица лучших параметров и CV/test метрик; график; краткое сравнение и ограничения.